# 使用BOCHA Web搜索API获取最新技术资讯

本笔记本演示如何使用BOCHA Web Search API，获取感兴趣领域的最新资讯。

**参考文档**: 
- [BOCHA开放平台](https://open.bochaai.com/)
- [API文档](https://bocha-ai.feishu.cn/wiki/RXEOw02rFiwzGSkd9mUcqoeAnNK)

**特点**:
- ✅ 简单易用，只需API Key
- ✅ 支持实时联网搜索（近10亿网页）
- ✅ 可选AI自动生成摘要
- ✅ 支持时间范围过滤
- ✅ 多模态混合搜索
- ✅ 语义重排序

## 步骤1: 导入必要的库

In [1]:
import json
import requests
import os
from dotenv import load_dotenv
from datetime import datetime
from typing import List, Dict, Any, Optional

print("✓ 库导入成功")

✓ 库导入成功


## 步骤2: 加载环境变量和配置

从 `.env` 文件中加载API密钥

In [2]:
# 加载环境变量
load_dotenv()

# 获取API凭证
BOCHA_API_KEY = os.getenv('BOCHA_API_KEY')

# API配置
API_URL = "https://api.bochaai.com/v1/web-search"

# 验证API密钥是否加载成功
if BOCHA_API_KEY:
    print(f"✓ API密钥已加载 (长度: {len(BOCHA_API_KEY)} 字符)")
    print(f"✓ API端点: {API_URL}")
else:
    print("✗ 警告: API密钥未找到，请检查.env文件")
    print("  请在.env文件中添加: BOCHA_API_KEY=your_api_key_here")

✓ API密钥已加载 (长度: 35 字符)
✓ API端点: https://api.bochaai.com/v1/web-search


## 步骤3: 定义感兴趣的技术领域

自定义你想要获取资讯的技术领域和时间范围

In [ ]:
# 定义感兴趣的技术领域
TECH_FIELDS = [
    "自动驾驶",
    "具身智能",
    "大模型",
    "人工智能"
]

# 定义重点关注的主题
FOCUS_TOPICS = [
    "特斯拉",
    "人形机器人",
    "AI大模型突破",
    "行业重要动态"
]

# 时间范围设置
# 可选: 'oneDay' (最近一天), 'oneWeek' (最近一周), 'oneMonth' (最近一月), 'oneYear' (最近一年)
SEARCH_FRESHNESS = "oneWeek"  # 最近一周

# 搜索配置
SEARCH_COUNT = 8  # 返回搜索结果数量
ENABLE_SUMMARY = True  # 是否开启AI生成摘要

print(f"关注领域: {', '.join(TECH_FIELDS)}")
print(f"重点主题: {', '.join(FOCUS_TOPICS)}")
print(f"时间范围: {SEARCH_FRESHNESS}")
print(f"结果数量: {SEARCH_COUNT}")
print(f"AI摘要: {'开启' if ENABLE_SUMMARY else '关闭'}")

关注领域: 自动驾驶, 具身智能, 大模型, 人工智能
重点主题: 特斯拉FSD, 人形机器人, AI大模型突破, 行业重要动态
时间范围: oneWeek
结果数量: 8
AI摘要: 开启


## 步骤4: 创建Web搜索客户端

封装BOCHA Web Search API

In [4]:
class BochaWebSearchClient:
    """
    BOCHA Web搜索客户端
    
    基于官方API文档: https://open.bochaai.com/
    """
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.api_url = "https://api.bochaai.com/v1/web-search"
        
    def _get_headers(self) -> Dict[str, str]:
        """获取请求头"""
        return {
            'Authorization': f'Bearer {self.api_key}',
            'Content-Type': 'application/json'
        }
    
    def search(self,
               query: str,
               freshness: str = "oneWeek",
               count: int = 8,
               summary: bool = False) -> Optional[Dict[str, Any]]:
        """
        执行Web搜索
        
        参数:
            query: 搜索查询
            freshness: 时间范围过滤 ('oneDay', 'oneWeek', 'oneMonth', 'oneYear')
            count: 返回结果数量
            summary: 是否开启AI生成摘要
        
        返回:
            搜索结果，格式:
            {
              "_type": "SearchResponse",
              "queryContext": {"originalQuery": "..."},
              "webPages": {
                "totalEstimatedMatches": 1000,
                "value": [
                  {
                    "id": "1",
                    "name": "标题",
                    "url": "https://...",
                    "siteName": "网站名",
                    "siteIcon": "https://...",
                    "snippet": "摘要",
                    "summary": "AI生成的总结",
                    "datePublished": "2025-01-01T00:00:00"
                  }
                ]
              }
            }
        """
        headers = self._get_headers()
        
        # 构建请求体（根据官方文档格式）
        body = {
            "query": query,
            "freshness": freshness,
            "count": count,
            "summary": summary
        }
        
        try:
            response = requests.post(self.api_url, json=body, headers=headers, timeout=120)
            
            if response.status_code == 200:
                return response.json()
            else:
                print(f"✗ API调用失败: {response.status_code}")
                print(f"响应: {response.text}")
                return None
        except Exception as e:
            print(f"✗ API调用错误: {e}")
            return None

print("✓ BochaWebSearchClient类已定义")

✓ BochaWebSearchClient类已定义


## 步骤5: 创建智能查询prompt

构建结构化的查询prompt

In [5]:
def create_search_query(fields: List[str], topics: List[str]) -> str:
    """
    创建搜索查询
    
    参数:
        fields: 技术领域列表
        topics: 关注主题列表
    
    返回:
        查询字符串
    """
    all_topics = fields + topics
    topics_str = " ".join(all_topics)
    
    # 简洁的查询格式，让搜索引擎自然匹配
    query = topics_str
    
    return query

print("✓ 查询prompt函数已定义")

✓ 查询prompt函数已定义


## 步骤6: 执行Web搜索

使用BOCHA Web Search API获取最新资讯

In [6]:
# 初始化客户端
if BOCHA_API_KEY:
    client = BochaWebSearchClient(BOCHA_API_KEY)
    print("✓ BOCHA Web搜索客户端初始化成功")
    
    # 创建查询
    query = create_search_query(TECH_FIELDS, FOCUS_TOPICS)
    
    print(f"\n{'='*80}")
    print("正在执行Web搜索...")
    print(f"{'='*80}\n")
    print(f"查询内容: {query}")
    print(f"时间范围: {SEARCH_FRESHNESS}\n")
    
    # 执行搜索
    search_result = client.search(
        query=query,
        freshness=SEARCH_FRESHNESS,
        count=SEARCH_COUNT,
        summary=ENABLE_SUMMARY
    )
    
    if search_result:
        print("\n✓ 搜索完成")
        
        # 显示基本信息
        if '_type' in search_result:
            print(f"  - 响应类型: {search_result['_type']}")
        
        query_context = search_result.get('queryContext', {})
        if query_context:
            original_query = query_context.get('originalQuery', '')
            if original_query:
                print(f"  - 原始查询: {original_query}")
        
        web_pages = search_result.get('webPages', {})
        if web_pages:
            total_results = web_pages.get('totalEstimatedMatches', 0)
            results_count = len(web_pages.get('value', []))
            if total_results:
                print(f"  - 预估匹配数: {total_results:,}")
            print(f"  - 返回结果数: {results_count}")
    else:
        print("\n✗ 搜索失败")
else:
    print("✗ 缺少API密钥，无法初始化客户端")
    client = None
    search_result = None

✓ BOCHA Web搜索客户端初始化成功

正在执行Web搜索...

查询内容: 自动驾驶 具身智能 大模型 人工智能 特斯拉FSD 人形机器人 AI大模型突破 行业重要动态
时间范围: oneWeek


✓ 搜索完成


## 步骤7: 展示搜索结果

显示搜索结果和AI生成的摘要

In [7]:
def display_search_results(result: Dict[str, Any]):
    """
    显示搜索结果（支持BOCHA API响应格式）
    """
    if not result:
        print("✗ 无搜索结果")
        return
    
    # BOCHA API响应格式: {"code": 200, "data": {...}}
    # 首先检查是否有code字段（BOCHA格式）
    if 'code' in result:
        if result['code'] != 200:
            print(f"✗ API返回错误: code={result['code']}, msg={result.get('msg', 'N/A')}")
            return
        
        # 提取data部分
        result = result.get('data', {})
        if not result:
            print("✗ 响应中没有data字段")
            return
    
    # 获取网页搜索结果
    web_pages = result.get('webPages', {})
    search_results = web_pages.get('value', [])
    
    if not search_results:
        print("✗ 未找到搜索结果")
        return
    
    # 显示搜索统计信息
    total_matches = web_pages.get('totalEstimatedMatches', 0)
    if total_matches:
        print(f"\n📊 预估匹配数: {total_matches:,}")
    
    print(f"\n🔍 搜索结果 (共{len(search_results)}条):\n")
    
    for i, item in enumerate(search_results, 1):
        try:
            name = item.get('name', 'N/A')
            url = item.get('url', 'N/A')
            site_name = item.get('siteName', '')
            snippet = item.get('snippet', '')
            summary = item.get('summary', '')
            date_published = item.get('datePublished', '')
            
            print(f"\n[{i}] 🌐 {name}")
            
            # 显示网站信息
            if site_name:
                print(f"   来源: {site_name}", end='')
                if date_published:
                    # 格式化日期显示
                    try:
                        date_str = date_published.split('T')[0] if 'T' in date_published else date_published
                        print(f" | 发布: {date_str}")
                    except:
                        print(f" | 发布: {date_published}")
                else:
                    print()
            
            # 显示原始摘要
            if snippet:
                # 清理摘要文本（去除多余的换行和空格）
                snippet_clean = ' '.join(snippet.split())
                snippet_text = snippet_clean[:200] + '...' if len(snippet_clean) > 200 else snippet_clean
                print(f"   摘要: {snippet_text}")
            
            # 显示AI生成的总结（如果有）
            if summary:
                # 清理总结文本
                summary_clean = ' '.join(summary.split())
                summary_text = summary_clean[:300] + '...' if len(summary_clean) > 300 else summary_clean
                print(f"   💡 AI总结: {summary_text}")
            
            # 显示链接
            print(f"   🔗 {url}")
            print("-"*80)
            
        except Exception as e:
            print(f"\n[{i}] ⚠️  解析错误: {e}")
            print(f"   原始数据: {json.dumps(item, ensure_ascii=False)[:200]}...")
            print("-"*80)

# 显示结果
if search_result:
    display_search_results(search_result)


📊 预估匹配数: 2,849,642

🔍 搜索结果 (共8条):


[1] 🌐 happyprince-CSDN博客
   来源: CSDN | 发布: 2025-11-07
   摘要: 摘要 2025年11月,全球AI领域迎来技术爆发期,涵盖机器人、大模型及多领域应用。小鹏、宇树推出新一代人形机器人,优必选获亿元订单;谷歌、科大讯飞等发布GPT-5、Gemini 3.0 Pro、星火
   💡 AI总结: 摘要 2025年11月,全球AI领域迎来技术爆发期,涵盖机器人、大模型及多领域应用。小鹏、宇树推出新一代人形机器人,优必选获亿元订单;谷歌、科大讯飞等发布GPT-5、Gemini 3.0 Pro、星火X1.5等大模型,多模态技术突破显著;AI应用深入视频翻译、医疗、教育及工业场景;太空算力设施加速布局;同时安全与伦理监管持续强化,推动技术合规发展。 关键词:人形机器人、大模型、多模态、AI Agent、太空算力、商业化应用、AI安全、垂直领域AI、算力硬件、视频生成 2025-11-07 07:00:00 1012 AI领域最新进展与挑战 技术突破:国内外AI模型在多个领域取得显著成果。国内方...
   🔗 https://happyprince.blog.csdn.net/
--------------------------------------------------------------------------------

[2] 🌐 熬夜到凌晨1点分析2025年自动驾驶究竟发展到什么地步 - 星创易联 - 博客园
   来源: 博客园 | 发布: 2025-11-07
   摘要: 哥们儿们,现在是凌晨1点12分,刚从公司加班回来,躺床上刷到小鹏的Iron机器人和特斯拉FSD V13的新闻,实在忍不住想跟大家聊聊自动驾驶这摊子事。这两天行业里又炸锅了,正好趁着还没困,给大家掰扯掰
   💡 AI总结: 哥们儿们,现在是凌晨1点12分,刚从公司加班回来,躺床上刷到小鹏的Iron机器人和特斯拉FSD V13的新闻,实在忍不住想跟大家聊聊自动驾驶这摊子事。这两天行业里又炸锅了,正好趁着还没困,给大家掰扯掰扯。 先说说这周发生了啥(11月初行业动态) 小鹏刚发布的Iron人形机器人,说实话我一开始以为又是
   🔗 https://www.cn

## 步骤8: 保存搜索结果

将搜索结果保存为JSON文件

In [8]:
def save_search_result(result: Dict[str, Any], filename: str = None) -> Optional[str]:
    """
    保存搜索结果到JSON文件
    
    参数:
        result: 搜索结果
        filename: 文件名（可选）
    
    返回:
        保存的文件路径
    """
    if not result:
        print("✗ 无结果可保存")
        return None
    
    # 生成文件名
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"tech_news_bocha_websearch_{timestamp}.json"
    
    try:
        # 保存完整的搜索结果
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        
        file_size = os.path.getsize(filename)
        print(f"\n✓ 结果已保存到: {filename}")
        print(f"  文件大小: {file_size:,} 字节")
        
        return filename
    except Exception as e:
        print(f"✗ 保存文件失败: {e}")
        return None

# 保存结果
if search_result:
    saved_file = save_search_result(search_result)
    if saved_file:
        print(f"\n可以使用以下代码重新加载数据:")
        print(f"with open('{saved_file}', 'r', encoding='utf-8') as f:")
        print(f"    result = json.load(f)")


✓ 结果已保存到: tech_news_bocha_websearch_20251110_143535.json
  文件大小: 20,471 字节

可以使用以下代码重新加载数据:
with open('tech_news_bocha_websearch_20251110_143535.json', 'r', encoding='utf-8') as f:
    result = json.load(f)


## 步骤9: 数据分析统计

对搜索结果进行统计分析

In [9]:
def analyze_search_result(result: Dict[str, Any]):
    """
    分析搜索结果（支持BOCHA API响应格式）
    """
    if not result:
        print("无法分析：结果为空")
        return
    
    # BOCHA API响应格式: {"code": 200, "data": {...}}
    if 'code' in result:
        if result['code'] != 200:
            print(f"✗ API返回错误: code={result['code']}, msg={result.get('msg', 'N/A')}")
            return
        result = result.get('data', {})
    
    print("\n" + "="*80)
    print(f"{'搜索结果统计分析':^76}")
    print("="*80)
    
    # 1. 搜索结果统计
    web_pages = result.get('webPages', {})
    search_results = web_pages.get('value', [])
    
    if search_results:
        print(f"\n📊 搜索结果统计:")
        print(f"   返回结果数: {len(search_results)} 条")
        
        total_matches = web_pages.get('totalEstimatedMatches', 0)
        if total_matches:
            print(f"   预估匹配数: {total_matches:,} 条")
        
        # 统计网站来源分布
        site_count = {}
        for item in search_results:
            site_name = item.get('siteName', '未知')
            if site_name:
                site_count[site_name] = site_count.get(site_name, 0) + 1
        
        if site_count:
            print("\n   信息源分布:")
            sorted_sites = sorted(site_count.items(), key=lambda x: x[1], reverse=True)
            for site, count in sorted_sites:
                print(f"     {site}: {count} 条")
        
        # 统计发布时间分布
        date_count = {}
        for item in search_results:
            date_published = item.get('datePublished', '')
            if date_published:
                # 提取日期部分
                date_only = date_published.split('T')[0] if 'T' in date_published else date_published
                date_count[date_only] = date_count.get(date_only, 0) + 1
        
        if date_count:
            print("\n   发布时间分布:")
            sorted_dates = sorted(date_count.items(), key=lambda x: x[0], reverse=True)
            for date, count in sorted_dates:
                print(f"     {date}: {count} 条")
        
        # 统计包含AI摘要的结果
        summary_count = sum(1 for item in search_results if item.get('summary'))
        if summary_count > 0:
            percentage = summary_count * 100 // len(search_results)
            print(f"\n   包含AI总结: {summary_count} 条 ({percentage}%)")
    else:
        print("\n未找到搜索结果")
    
    print("\n" + "="*80)

# 执行分析
if search_result:
    analyze_search_result(search_result)


                                  搜索结果统计分析                                  

📊 搜索结果统计:
   返回结果数: 8 条
   预估匹配数: 2,849,642 条

   信息源分布:
     博客园: 2 条
     CSDN: 1 条
     搜狐: 1 条
     股吧: 1 条
     证券之星股票频道: 1 条
     中华网新闻频道: 1 条
     新榜: 1 条

   发布时间分布:
     2025-11-10: 1 条
     2025-11-07: 4 条
     2025-11-04: 2 条
     2025-11-03: 1 条

   包含AI总结: 8 条 (100%)



## 步骤10: 完整流程封装

将以上步骤封装成一个完整的函数

In [10]:
def acquire_latest_news_bocha(
    api_key: str,
    fields: List[str],
    focus_topics: List[str],
    freshness: str = "oneWeek",
    count: int = 8,
    enable_summary: bool = True,
    save_to_file: bool = True,
    show_analysis: bool = True
) -> Optional[Dict[str, Any]]:
    """
    完整的新闻获取流程（使用BOCHA Web搜索）
    
    参数:
        api_key: BOCHA API Key
        fields: 技术领域列表
        focus_topics: 重点关注主题列表
        freshness: 时间范围 ('oneDay', 'oneWeek', 'oneMonth', 'oneYear')
        count: 返回结果数量
        enable_summary: 是否开启AI生成摘要
        save_to_file: 是否保存到文件
        show_analysis: 是否显示统计分析
    
    返回:
        搜索结果
    """
    print("\n" + "="*80)
    print(f"{'开始获取最新技术资讯 (BOCHA Web搜索)':^76}")
    print("="*80)
    
    # 1. 初始化客户端
    print("\n[1/4] 初始化Web搜索客户端...")
    client = BochaWebSearchClient(api_key)
    
    # 2. 创建查询
    print("\n[2/4] 构建搜索查询...")
    query = create_search_query(fields, focus_topics)
    print(f"查询: {query}")
    
    # 3. 执行搜索
    print("\n[3/4] 执行Web搜索...")
    result = client.search(
        query=query,
        freshness=freshness,
        count=count,
        summary=enable_summary
    )
    
    if not result:
        print("\n✗ 流程终止：搜索失败")
        return None
    
    # 4. 显示结果
    print("\n[4/4] 显示搜索结果...")
    display_search_results(result)
    
    # 5. 保存文件
    if save_to_file:
        save_search_result(result)
    
    # 6. 统计分析
    if show_analysis:
        analyze_search_result(result)
    
    print("\n" + "="*80)
    print(f"{'流程完成！':^76}")
    print("="*80)
    
    return result

print("✓ 完整流程函数已定义")
print("\n快速使用示例:")
print("""
# 获取最新技术资讯
result = acquire_latest_news_bocha(
    api_key=BOCHA_API_KEY,
    fields=["自动驾驶", "具身智能", "大模型"],
    focus_topics=["特斯拉FSD", "人形机器人"],
    freshness="oneWeek",  # 最近一周
    count=8,  # 返回8条结果
    enable_summary=True,  # 开启AI摘要
    save_to_file=True,
    show_analysis=True
)
""")

✓ 完整流程函数已定义

快速使用示例:

# 获取最新技术资讯
result = acquire_latest_news_bocha(
    api_key=BOCHA_API_KEY,
    fields=["自动驾驶", "具身智能", "大模型"],
    focus_topics=["特斯拉FSD", "人形机器人"],
    freshness="oneWeek",  # 最近一周
    count=8,  # 返回8条结果
    enable_summary=True,  # 开启AI摘要
    save_to_file=True,
    show_analysis=True
)



## 总结

本笔记本演示了如何使用BOCHA Web Search API获取最新技术资讯，包括：

1. ✅ 环境配置和API密钥加载
2. ✅ 自定义技术领域和关注主题
3. ✅ 创建Web搜索客户端
4. ✅ 构建智能搜索查询
5. ✅ 执行实时联网搜索
6. ✅ 展示搜索结果和AI摘要
7. ✅ 保存搜索结果
8. ✅ 数据统计分析
9. ✅ 完整流程封装

### 核心优势

#### 1. 简单易用
- **只需API Key**: 无需复杂配置
- **RESTful API**: 标准HTTP请求
- **即时可用**: 开箱即用

#### 2. 强大功能
- **海量数据**: 近10亿网页索引
- **实时搜索**: 获取最新信息
- **AI增强**: 可选AI生成摘要
- **时间过滤**: 支持oneDay/oneWeek/oneMonth/oneYear
- **语义重排**: 提高结果相关性
- **多模态**: 支持文本、图片、视频等

#### 3. 响应结构
- **SearchResponse**: 标准的搜索响应格式
- **webPages**: 包含搜索结果列表
- **每个结果包含**:
  - `name`: 标题
  - `url`: 链接
  - `siteName`: 网站名称
  - `siteIcon`: 网站图标
  - `snippet`: 原始摘要
  - `summary`: AI生成的总结（如果开启）
  - `datePublished`: 发布时间

### 使用建议

#### 获取API Key

1. 访问 [BOCHA开放平台](https://open.bochaai.com/)
2. 注册并登录账号
3. 在控制台获取API Key

#### 配置示例

在 `.env` 文件中添加：
```bash
# BOCHA Web Search
BOCHA_API_KEY=your_api_key_here
```

#### 基础用法

```python
# 获取最新资讯（带AI摘要）
result = acquire_latest_news_bocha(
    api_key=BOCHA_API_KEY,
    fields=["自动驾驶", "具身智能", "大模型"],
    focus_topics=["特斯拉FSD", "人形机器人"],
    freshness="oneWeek",  # 最近一周
    count=8,  # 返回8条结果
    enable_summary=True  # 开启AI摘要
)
```

### 参数说明

| 参数 | 类型 | 说明 | 默认值 |
|------|------|------|--------|
| api_key | str | BOCHA API Key | 必填 |
| fields | List[str] | 技术领域列表 | 必填 |
| focus_topics | List[str] | 重点主题列表 | 必填 |
| freshness | str | 时间范围 | "oneWeek" |
| count | int | 返回结果数量 | 8 |
| enable_summary | bool | 是否开启AI摘要 | True |
| save_to_file | bool | 是否保存文件 | True |
| show_analysis | bool | 是否显示分析 | True |

### 时间范围选项

- `oneDay`: 最近一天
- `oneWeek`: 最近一周（推荐用于获取最新动态）
- `oneMonth`: 最近一月
- `oneYear`: 最近一年

### 与其他方案对比

| 特性 | BOCHA Web Search | 千帆AI搜索 | 腾讯混元 | 秘塔META |
|------|------------------|-----------|----------|----------|
| 搜索方式 | 多模态混合搜索 | AI+百度搜索 | AI联网搜索 | 直接搜索引擎 |
| 配置复杂度 | 低（仅API Key） | 低（仅API Key） | 中 | 低 |
| 数据规模 | 近10亿网页 | 大 | 中 | 中 |
| AI摘要 | 可选 | 是 | 是 | 否 |
| 时间过滤 | 原生支持 | 原生支持 | AI理解 | 查询词 |
| 语义重排 | 是 | 否 | 否 | 否 |
| 多模态 | 是 | 部分 | 否 | 否 |
| 搜索质量 | 高 | 高（百度） | 高 | 中 |
| 适用场景 | 专业搜索+AI增强 | 权威来源+总结 | 快速分析 | 原始采集 |

### 注意事项

1. **API限制**
   - 请查看BOCHA平台的具体限额
   - 合理控制请求频率
   - 注意API调用成本

2. **成本控制**
   - AI摘要功能可能产生额外成本
   - 合理设置count参数
   - 根据需求选择是否开启summary

3. **数据质量**
   - AI摘要可能存在误差
   - 重要信息建议查看原文
   - 参考多个来源交叉验证

4. **时间范围**
   - oneWeek适合获取最新动态
   - oneMonth适合了解近期趋势
   - 时间范围越短，结果越精准

### 最佳实践

1. **优化查询**
   - 使用具体的关键词组合
   - 利用多模态搜索能力
   - 合理设置时间范围

2. **结果验证**
   - 检查来源网站的权威性
   - 对比snippet和summary
   - 访问原文链接核实

3. **效率提升**
   - 使用合适的count参数
   - 定期保存搜索结果
   - 利用分析功能了解趋势

### 下一步扩展

- ✅ 添加定时任务，自动获取资讯
- ✅ 实现多次搜索结果的对比分析
- ✅ 集成通知功能（邮件/企业微信）
- ✅ 构建个性化推荐系统
- ✅ 添加情感分析和趋势预测
- ✅ 与其他数据源整合对比
- ✅ 利用语义重排优化结果